In [0]:
invoice_data = spark.read.format('csv')\
               .option('header','true')\
                    .option('inferSchema', 'true').load("/Volumes/dev/spark_db/datasets/spark_programming/data/invoices.csv")

In [0]:
display(invoice_data)

####2. Calculate total_value for each invoice line item


In [0]:
from pyspark.sql.functions import round,col
order_value_per_line =  invoice_data.withColumn('LineAmount', round(col('UnitPrice')* col('Quantity'),0))

In [0]:
order_value_per_line.filter(col('InvoiceNo')=='536370').display()

In [0]:
from pyspark.sql.functions import col,sum,desc,count_distinct,avg

invoice_data_sum = order_value_per_line\
    .groupBy(col('InvoiceNo'))\
        .agg(round(sum(col('LineAmount')),0)\
            .alias('InvoiceTotal'),\
                count_distinct(col('StockCode'))\
                    .alias('LineItemCount') )

In [0]:
invoice_data_sum.display()

In [0]:
invoice_data_invoice_item_info = (
    invoice_data_sum.withColumn('avgValuePerLinePerInvoice',
    round(col('InvoiceTotal')/
          col('LineItemCount'),0))
)

In [0]:
# invoice_data_avg_amount_per_line_per_invoice=(invoice_data_sum
#     .agg(
#         round(avg(
#             col('InvoiceTotal')/ col('LineItemCount')),0)
#             .alias('AvgLineAmountPerInvoice')
#          )
# )

In [0]:
invoice_data_invoice_item_info.display()

####3. Perform the following exploratory analysis on invoices data
1. Can we make invoice numbers a numeric field?
2. Analyize quantity to identify potentially invalid records
3. Analyze unit price to identify potentially invalid records


In [0]:
from pyspark.sql.functions import when
invoice_data_invoice_item_info_eda =(
invoice_data_invoice_item_info.withColumns(
                {
                    #"InvoiceNo" : col("InvoiceNo").cast('long'),
                    "InvoiceTotal_amended" : when(col("InvoiceTotal")<0 , 0).otherwise(col("InvoiceTotal")),
                    "avgValuePerLinePerInvoice_amended" : when(col("avgValuePerLinePerInvoice")<0 , 0).otherwise(col("avgValuePerLinePerInvoice"))

                    
                }

) 

                )

In [0]:
invoice_data_invoice_item_info_eda.display()

3.1 Analyze using dataframe summary

Available statistics are:
```
  count - mean - stddev - min - max - approximate percentiles
```

In [0]:
invoice_data_invoice_item_info_eda.summary().select('summary', 'InvoiceNo', 'InvoiceTotal').display()

3.2 Using sql functions

In [0]:
#what is 90% percentile
from pyspark.sql.functions import min,max, percentile

invoice_data_invoice_item_stats =( 
        invoice_data_invoice_item_info_eda.select(
                    min('InvoiceTotal_amended'),\
                    percentile("InvoiceTotal_amended",0.9)

))

invoice_data_invoice_item_stats.display()